# FuzNex reproducible video replay

This is a thin Kaggle runner for the versioned `ai/` package in GitHub—not a second inference implementation. Enable **GPU** in Kaggle Settings for experiments. Outputs are detection-only: no identity, gallery, embedding, or face-to-person association is emitted.

Use only an openly licensed sample or a private, consented video. Do not upload a biometric gallery, enrollment images, or a public identity dataset.


In [ ]:
# Read-only checkout. Set this to a reviewed commit SHA when comparing experiments.
FUZNEX_REF = 'integrate/portable-ai'
!git clone --filter=blob:none --no-checkout https://github.com/harrsh-here/fuznex-smart-surveillance.git fuznex-smart-surveillance
%cd fuznex-smart-surveillance
!git fetch origin $FUZNEX_REF --depth 1
!git checkout --detach FETCH_HEAD
!git rev-parse HEAD
!git status --short  # expected: no output (read-only source checkout)


In [ ]:
# Install only the runtime used by the repository. Kaggle's preinstalled CUDA PyTorch is used.
!python -m pip install -q pytest -r ai/requirements/yolo.txt
!python -m unittest discover -s tests -p 'test_*.py' -v


## Kaggle inputs

Attach two private Kaggle datasets (or use paths to files already attached):

- **Video:** one openly licensed sample clip, or a private consented clip.
- **Models:** only locally attached artifacts whose provenance/licensing you have reviewed against `ai/model-manifest.json`. The repo never downloads weights. `det_500m.onnx` is checked against the manifest hash; YOLO11n currently has no pinned manifest hash, so record its source and digest before accepting a result.

SFace may be attached for future local experiments, but this notebook never accepts a private gallery and does not perform recognition.


In [ ]:
from pathlib import Path
VIDEO = Path('/kaggle/input/YOUR_VIDEO_DATASET/clip.mp4')
MODELS = Path('/kaggle/input/YOUR_APPROVED_MODELS')
YOLO = MODELS / 'yolo11n.pt'
SCRFD = MODELS / 'det_500m.onnx'
OUTPUT = Path('/kaggle/working/fuznex-replay')
assert VIDEO.is_file(), 'Attach an openly licensed or consented video and set VIDEO.'
assert YOLO.is_file() and SCRFD.is_file(), 'Attach reviewed local model artifacts and set MODELS.'


In [ ]:
# Same input + same revision + same parameters gives a repeatable replay sequence.
!python -m ai.video_replay --input "{VIDEO}" --output-dir "{OUTPUT}" --yolo "{YOLO}" --scrfd "{SCRFD}" --device cuda --input-size 320 --confidence 0.25 --max-frames 300


In [ ]:
# Preview the annotated derivative and inspect privacy-safe session artifacts.
from IPython.display import Video, display
sessions = sorted(OUTPUT.glob('spectra-*'))
assert sessions, 'Replay did not create a session; read the command output above.'
session = sessions[-1]
display(Video(str(session / 'annotated.mp4'), embed=True, width=900))
print((session / 'session-summary.json').read_text())
print((session / 'provenance.json').read_text())
# events.jsonl and detections.csv are exported for metrics/comparison, never as an identity dataset.


## Reproducibility and honest limits

Record the printed commit SHA, input file digest, model digests, Kaggle accelerator, and parameters with each experiment. Results measure decode + processing + annotation on Kaggle, not camera/network latency and not local-PC performance. A missing or mismatched required model causes a blocked result—there is no mock fallback. A public clip without consented ground truth cannot validate identity accuracy.
